https://archive.ics.uci.edu/dataset/211/communities+and+crime+unnormalized

In [348]:
#%pip install ucimlrepo
#%pip install seaborn

In [349]:
from ucimlrepo import fetch_ucirepo 

import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [350]:
# Carrega o dataset
communities_and_crime_unnormalized = fetch_ucirepo(id=211) 

# X recebe as variáveis de entrada (features)
X = communities_and_crime_unnormalized.data.features 

# y recebe as variáveis alvo (targets)
y = communities_and_crime_unnormalized.data.targets

In [351]:
# Concatena as features (X) e os alvos (y) em um único DataFrame
df = pd.concat([X, y], axis=1)

In [352]:
print("shape:", df.shape)
print("columns:", df.columns.tolist())
print("total missing:", df.isnull().sum().sum())
print(df.isnull().sum().sort_values(ascending=False).head(30))

shape: (2215, 143)
columns: ['State', 'pop', 'perHoush', 'pctBlack', 'pctWhite', 'pctAsian', 'pctHisp', 'pct12-21', 'pct12-29', 'pct16-24', 'pct65up', 'persUrban', 'pctUrban', 'medIncome', 'pctWwage', 'pctWfarm', 'pctWdiv', 'pctWsocsec', 'pctPubAsst', 'pctRetire', 'medFamIncome', 'perCapInc', 'whitePerCap', 'blackPerCap', 'NAperCap', 'asianPerCap', 'otherPerCap', 'hispPerCap', 'persPoverty', 'pctPoverty', 'pctLowEdu', 'pctNotHSgrad', 'pctCollGrad', 'pctUnemploy', 'pctEmploy', 'pctEmployMfg', 'pctEmployProfServ', 'pctOccupManu', 'pctOccupMgmt', 'pctMaleDivorc', 'pctMaleNevMar', 'pctFemDivorc', 'pctAllDivorc', 'persPerFam', 'pct2Par', 'pctKids2Par', 'pctKids-4w2Par', 'pct12-17w2Par', 'pctWorkMom-6', 'pctWorkMom-18', 'kidsBornNevrMarr', 'pctKidsBornNevrMarr', 'numForeignBorn', 'pctFgnImmig-3', 'pctFgnImmig-5', 'pctFgnImmig-8', 'pctFgnImmig-10', 'pctImmig-3', 'pctImmig-5', 'pctImmig-8', 'pctImmig-10', 'pctSpeakOnlyEng', 'pctNotSpeakEng', 'pctLargHousFam', 'pctLargHous', 'persPerOccupHous',

In [353]:
# Obtém número de linhas e colunas do DataFrame
n_rows, n_cols = df.shape

# Conta valores nulos por coluna
missing_sum = df.isnull().sum()

# Filtra apenas colunas que possuem pelo menos 1 valor faltante
cols_with_missing = missing_sum[missing_sum > 0]

# Imprime total geral de valores faltantes no dataset
print(f"Total de valores faltantes: {missing_sum.sum()}")

# Imprime quantidade de colunas que possuem valores faltantes
print(f"Colunas afetadas: {len(cols_with_missing)}")

# Calcula percentual de nulos por coluna e mostra as 5 com maior proporção
print(f"colunas com mais nulos (%):")
print((cols_with_missing / n_rows * 100).sort_values(ascending=False).head())

Total de valores faltantes: 42147
Colunas afetadas: 39
colunas com mais nulos (%):
numPolice            84.514673
policePerPop         84.514673
policeField          84.514673
policeCalls          84.514673
policeFieldPerPop    84.514673
dtype: float64


Colunas com mais de 80% de valores faltantes foram removidas por apresentarem informação insuficiente para imputação confiável, reduzindo risco de viés e overfitting. Colunas com menos de 10% de ausência foram mantidas e imputadas via mediana.

In [354]:
# colunas com >80% de nulos
missing_cols = df.columns[df.isnull().mean() > 0.8]

# dropar essas colunas
df = df.drop(columns=missing_cols)

# conferir novo shape
print(df.shape)

(2215, 121)


In [355]:
#inprimir 10 variáveis com mais valores faltantes
print(df.isnull().sum().sort_values(ascending=False).head(10))

violentPerPop    221
rapes            208
rapesPerPop      208
nonViolPerPop     97
arsonsPerPop      91
arsons            91
assaultPerPop     13
assaults          13
larcPerPop         3
autoTheft          3
dtype: int64


Imputação da mediana no resto dos dados faltante

In [356]:
# remover linhas onde o target é nulo
df = df.dropna(subset=['violentPerPop'])

# conferir
print("Novo shape:", df.shape)
print("NAs no target:", df['violentPerPop'].isnull().sum())
print("NAs no total:", df.isnull().sum().sum())

Novo shape: (1994, 121)
NAs no target: 0
NAs no total: 283


In [357]:
X = df.drop(columns=['violentPerPop'])
y = df['violentPerPop']

print(X.shape, y.shape)

(1994, 120) (1994,)


In [358]:
from sklearn.impute import SimpleImputer

num_cols = X.select_dtypes(include=['number']).columns

imputer = SimpleImputer(strategy='median')
X[num_cols] = imputer.fit_transform(X[num_cols])

print("Missing após imputação X:", X.isnull().sum().sum())
print("Missing após imputação y:", y.isnull().sum().sum())

Missing após imputação X: 0
Missing após imputação y: 0


In [359]:
df = pd.concat([X,y],axis=1)

In [360]:
from sklearn.cluster import KMeans
import numpy as np

goal_col = 'violentPerPop'

kmeans = KMeans(n_clusters=2, random_state=42)
df['violent_class'] = kmeans.fit_predict(df[[goal_col]])

centers = kmeans.cluster_centers_.flatten()
order = np.argsort(centers)

mapping = {order[i]: i for i in range(2)}
df['violent_class'] = df['violent_class'].map(mapping)

In [361]:
crime_cols = [
    'murders','rapes','robberies','assaults','burglaries',
    'larcenies','autoTheft','arsons',
    'autoTheftPerPop','arsonsPerPop',
    'murdPerPop','rapesPerPop','robbbPerPop',
    'assaultPerPop','burglPerPop','larcPerPop',
    'violentPerPop','nonViolPerPop'
]

df = df.drop(columns=[c for c in crime_cols if c in df.columns])

print("Novo shape:", df.shape)

Novo shape: (1994, 104)


In [362]:
non_predictive = [
    'State'
]

df = df.drop(columns=[c for c in non_predictive if c in df.columns])

print("Novo shape:", df.shape)

Novo shape: (1994, 103)


In [363]:
# separar X e y
X = df.drop(columns=['violent_class'])
y = df['violent_class']

# split estratificado
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

from sklearn.metrics import classification_report, confusion_matrix

In [364]:
#Arvore de decisão
from sklearn import tree

model = tree.DecisionTreeClassifier(random_state=42)
model = model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

[[287  41]
 [ 30  41]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.88      0.89       328
           1       0.50      0.58      0.54        71

    accuracy                           0.82       399
   macro avg       0.70      0.73      0.71       399
weighted avg       0.83      0.82      0.83       399



In [365]:
#KNN
from sklearn.neighbors import KNeighborsClassifier
import math

k = int(math.sqrt(len(df)))
model = KNeighborsClassifier(n_neighbors=k)
model = model.fit(X_train,y_train)

y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

[[325   3]
 [ 57  14]]

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.99      0.92       328
           1       0.82      0.20      0.32        71

    accuracy                           0.85       399
   macro avg       0.84      0.59      0.62       399
weighted avg       0.85      0.85      0.81       399



In [366]:
#SVM
from sklearn import svm

model = svm.SVC(kernel='rbf')
model = model.fit(X_train,y_train)

y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

[[326   2]
 [ 61  10]]

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.99      0.91       328
           1       0.83      0.14      0.24        71

    accuracy                           0.84       399
   macro avg       0.84      0.57      0.58       399
weighted avg       0.84      0.84      0.79       399



In [367]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

model = LinearDiscriminantAnalysis()
model = model.fit(X_train,y_train)

y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

[[313  15]
 [ 22  49]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.95      0.94       328
           1       0.77      0.69      0.73        71

    accuracy                           0.91       399
   macro avg       0.85      0.82      0.84       399
weighted avg       0.90      0.91      0.91       399



In [368]:
from sklearn.ensemble import AdaBoostClassifier

model = AdaBoostClassifier(n_estimators=100, random_state=42)
model = model.fit(X_train,y_train)

y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

[[310  18]
 [ 24  47]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.95      0.94       328
           1       0.72      0.66      0.69        71

    accuracy                           0.89       399
   macro avg       0.83      0.80      0.81       399
weighted avg       0.89      0.89      0.89       399

